# Analytics Build: Tourism Activity, Recovery, and Economic Alignment

## Purpose

This notebook builds the Analytics layer for ProvInsights PH Tourism from Gold-layer inputs. It produces five persisted Delta tables in `tuklas_dev`.`05_analytics` that answer the project's business questions (BQ1–BQ2) and their analytical questions (AQ).

## Analytical source of truth

The Analytics layer is designed to answer the **current approved ProvInsights Business and Analytical Questions**. It surfaces provinces/reporting units and patterns that warrant closer investigation. It does **not** claim causality, prescribe investments, or invent thresholds not defined by the approved questions.

## Prerequisites

Gold tables must exist and be populated:

- `tuklas_dev`.`04_gold`.`dim_province`: 100 reporting units
- `tuklas_dev`.`04_gold`.`dim_year`: 2019–2024
- `tuklas_dev`.`04_gold`.`fact_province_year`: 600 rows (100 units × 6 years)

The Gold notebooks (`00_gold_setup`, `01_gold_dim_province`, `02_gold_dim_year`, `03_gold_fact_province_year`) must have completed successfully.

## Outputs

Five persisted Delta tables in `tuklas_dev`.`05_analytics`:

| Table | Grain | Key | BQ/AQ |
| --- | --- | --- | --- |
| `tourism_activity` | reporting unit × year | (`province_psgc`, `year`) | BQ1-AQ1.1 |
| `tourism_concentration` | year | (`year`) | BQ1-AQ1.1 |
| `tourism_recovery` | reporting unit × comparison year | (`province_psgc`, `baseline_year`, `comparison_year`) | BQ1-AQ1.2 |
| `traveler_segment_recovery` | reporting unit (2023 below-2019 cohort) | (`province_psgc`, `baseline_year`, `comparison_year`) | BQ1-AQ1.3 |
| `tourism_economy` | reporting unit | (`province_psgc`, `baseline_year`, `comparison_year`) | BQ2-AQ2.1, BQ2-AQ2.2, BQ2-AQ2.3 |

## Traceability

Every Analytics table carries five lineage columns, populated identically across all five tables in a single run:

* `_analytics_run_id` — unique run identifier
* `_gold_fact_version` — Delta version of `fact_province_year` at build time
* `_gold_dim_province_version` — Delta version of `dim_province` at build time
* `_gold_dim_year_version` — Delta version of `dim_year` at build time
* `_processed_at` — single processing timestamp for the entire run

## Build flow

**A** Overview → **B** Configuration & Gold snapshot → **C** Schema setup → **D** Gold input validation → **E** Tourism activity and concentration → **F** Tourism recovery → **G** Traveler-segment recovery → **H** Tourism and economic alignment → **I** Cross-table consistency → **J** Execution summary

Each phase builds candidate results in a temporary view, runs blocking checks, and only then replaces the persisted table. If a blocking check fails, execution stops before the table is replaced.

## Rerun behavior

All tables use `CREATE OR REPLACE TABLE`, so the notebook is idempotent. The five table writes are separate Delta operations, **not** one atomic transaction: if execution stops after some tables were replaced, the completion summary reports which outputs completed and which did not.

## Analytical caveats

- Tourism activity means reported overnight travelers in accommodation establishments.
- Recovery means reported overnight tourism activity relative to 2019.
- Missing values remain NULL — never replaced with zero.
- Invalid or missing denominators produce NULL derived metrics.
- Ineligible rows do not receive valid ranks or classifications.
- Constant-price A&F GVA is used for growth; current-price A&F GVA and GDP for share of GDP.
- A&F GVA is an economic proxy and may contain activity not caused by tourism.
- Tourism/economic co-movement is association, not causation.
- `above_2019` does not automatically mean emerging destination.

## Phase B: Configuration and Gold snapshot

**Purpose:** Set up run configuration, capture Gold Delta versions, create **pinned temporary views** using `VERSION AS OF` so all transformations read a consistent Gold snapshot, and initialise validation helpers.

**Critical:** The version snapshot enforces consistency — it is not just printed for documentation. All downstream SQL uses the pinned views, not the live Gold tables.

In [0]:
%python
from datetime import datetime
import uuid

CATALOG = "tuklas_dev"
GOLD_SCHEMA = "04_gold"
ANALYTICS_SCHEMA = "05_analytics"
DQ_SCHEMA = "06_data_quality"
NOTEBOOK_NAME = "01_build_analytics"
run_id = f"analytics_{datetime.now().strftime('%Y%m%d_%H%M%S')}_{uuid.uuid4().hex[:8]}"

BASELINE_YEAR = 2019
RECOVERY_YEARS = [2022, 2023]
HEADLINE_YEAR = 2023

# ── Gold Delta version snapshot ──────────────────────────────────────────────
gold_versions = {}
for tbl in ['dim_province', 'dim_year', 'fact_province_year']:
    try:
        v = spark.sql(f"DESCRIBE HISTORY `{CATALOG}`.`{GOLD_SCHEMA}`.`{tbl}`").orderBy("version", ascending=False).limit(1).select("version").first()[0]
        gold_versions[tbl] = int(v)
    except Exception as e:
        gold_versions[tbl] = None
        print(f"ERROR: Could not get Delta version for {tbl}: {e}")

print(f"Run ID: {run_id}")
print(f"Gold versions: {gold_versions}")

# ── Pinned temporary views using VERSION AS OF ───────────────────────────────
# These views enforce Gold snapshot consistency for the entire run.
for tbl, view_name in [('dim_province', 'v_gold_dim_province'),
                        ('dim_year', 'v_gold_dim_year'),
                        ('fact_province_year', 'v_gold_fact')]:
    v = gold_versions[tbl]
    if v is None:
        raise AssertionError(f"BLOCK FAILURE: Cannot capture Gold Delta version for {tbl} — run is not traceable")
    spark.sql(
        f"""CREATE OR REPLACE TEMPORARY VIEW {view_name} AS
        SELECT * FROM `{CATALOG}`.`{GOLD_SCHEMA}`.`{tbl}` VERSION AS OF {v}"""
    )
    print(f"  Pinned {view_name} → version {v}")

# Single processing timestamp for the entire run
processed_at = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Lineage temp view — CROSS JOINed into every candidate view
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW v_lineage AS
SELECT
  '{run_id}' AS _analytics_run_id,
  {gold_versions['fact_province_year']} AS _gold_fact_version,
  {gold_versions['dim_province']} AS _gold_dim_province_version,
  {gold_versions['dim_year']} AS _gold_dim_year_version,
  TIMESTAMP('{processed_at}') AS _processed_at
""")
print(f"  Lineage view created (processed_at={processed_at})")

# ── Validation helpers ───────────────────────────────────────────────────────
checks = []           # list of dicts: {name, severity, passed, details}
output_status = {}    # dict: table_name → {status, row_count, delta_version, error}

BLOCK_COUNT = 0
BLOCK_FAILURES = 0
REVIEW_COUNT = 0

def record_check(name, severity, passed, details=""):
    """Record a validation check. severity = 'BLOCK' or 'REVIEW'."""
    global BLOCK_COUNT, BLOCK_FAILURES, REVIEW_COUNT
    checks.append({"name": name, "severity": severity, "passed": passed, "details": details})
    if severity == "BLOCK":
        BLOCK_COUNT += 1
        if not passed:
            BLOCK_FAILURES += 1
    elif severity == "REVIEW":
        if not passed:
            REVIEW_COUNT += 1
    status = "PASS" if passed else ("BLOCK FAIL" if severity == "BLOCK" else "REVIEW")
    print(f"  [{status}] {name}: {details}")

def record_output(table, status, row_count=None, delta_version=None, error=None):
    """Record output table persistence status."""
    output_status[table] = {
        "status": status,
        "row_count": row_count,
        "delta_version": delta_version,
        "error": error
    }
    print(f"  OUTPUT {table}: {status} (rows={row_count}, v={delta_version})")

def assert_block(name, condition, details=""):
    """Assert a BLOCK check. Raises AssertionError if condition is False."""
    record_check(name, "BLOCK", condition, details)
    if not condition:
        raise AssertionError(f"BLOCK FAILURE: {name} — {details}")

print("Configuration complete.")

## Phase C: Schema setup

**Purpose:** Create the destination schema `tuklas_dev`.`05_analytics` so it is ready to receive the five Analytics tables.

**Rerun:** `CREATE SCHEMA IF NOT EXISTS` is idempotent.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS `tuklas_dev`.`05_analytics`

## Phase D: Gold input validation

**Purpose:** Verify that the three pinned Gold source views exist with expected row counts, unique natural keys, and all columns required by the Analytics layer. These are BLOCK checks: if any fails, the notebook stops before building any Analytics table.

All checks use the pinned views (`v_gold_dim_province`, `v_gold_dim_year`, `v_gold_fact`), not the live Gold tables.

### D1: Validate dim_province

Checks: 100 rows, unique non-NULL 10-digit `province_psgc`, non-NULL `province_name`/`unit_type`/`region_psgc`/`region_name`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_d1 AS
SELECT
  CASE WHEN COUNT(*) != 100 THEN 1 ELSE 0 END
  + CASE WHEN COUNT(DISTINCT province_psgc) != COUNT(province_psgc) THEN 1 ELSE 0 END
  + CASE WHEN COUNT(*) - COUNT(province_psgc) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN province_psgc NOT RLIKE '^[0-9]{10}$' THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN COUNT(*) - COUNT(province_name) > 0 THEN 1 ELSE 0 END
  + CASE WHEN COUNT(*) - COUNT(unit_type) > 0 THEN 1 ELSE 0 END
  + CASE WHEN COUNT(*) - COUNT(region_psgc) > 0 THEN 1 ELSE 0 END
  + CASE WHEN COUNT(*) - COUNT(region_name) > 0 THEN 1 ELSE 0 END
  AS fail_count
FROM v_gold_dim_province

### D2: Validate dim_year

Checks: exact year set {2019–2024}, unique non-NULL `year`, correct flag columns.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_d2 AS
SELECT
  CASE WHEN SORT_ARRAY(COLLECT_LIST(year)) != ARRAY(2019, 2020, 2021, 2022, 2023, 2024) THEN 1 ELSE 0 END
  + CASE WHEN COUNT(DISTINCT year) != COUNT(year) OR COUNT(*) - COUNT(year) > 0 THEN 1 ELSE 0 END
  + CASE WHEN COUNT(*) != 6 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN is_baseline_year IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN is_recovery_year IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN year = 2019 AND is_baseline_year != true THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN year NOT IN (2022, 2023) AND is_recovery_year = true THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  AS fail_count
FROM v_gold_dim_year

### D3: Validate fact_province_year

Checks: 600 rows (100 × 6), all `province_psgc` exist in `dim_province`, all `year` exist in `dim_year`, unique composite key, key traveler/economy columns present.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_d3 AS
WITH fk_province AS (
  SELECT COUNT(*) AS missing
  FROM v_gold_fact f
  LEFT JOIN v_gold_dim_province d ON f.province_psgc = d.province_psgc
  WHERE d.province_psgc IS NULL
),
fk_year AS (
  SELECT COUNT(*) AS missing
  FROM v_gold_fact f
  LEFT JOIN v_gold_dim_year y ON f.year = y.year
  WHERE y.year IS NULL
)
SELECT
  CASE WHEN (SELECT COUNT(*) FROM v_gold_fact) != 600 THEN 1 ELSE 0 END
  + CASE WHEN (SELECT missing FROM fk_province) > 0 THEN 1 ELSE 0 END
  + CASE WHEN (SELECT missing FROM fk_year) > 0 THEN 1 ELSE 0 END
  + CASE WHEN (SELECT COUNT(*) FROM (SELECT province_psgc, year FROM v_gold_fact GROUP BY province_psgc, year HAVING COUNT(*) > 1) dup) > 0 THEN 1 ELSE 0 END
  + CASE WHEN (SELECT COUNT(*) - COUNT(traveler_value_status) FROM v_gold_fact) > 0 THEN 1 ELSE 0 END
  AS fail_count

### D4: Required column availability

Checks that every column the Analytics layer needs exists in `fact_province_year`. Previously missing fields (`foreign_includes_overseas_filipinos`, `derived_values`, `per_capita_recomputed`) are verified against the current Gold implementation.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_d4 AS
WITH col_list AS (
  SELECT array('province_psgc','year','total_travelers','domestic_travelers','foreign_travelers',
    'overseas_filipinos','traveler_value_status','foreign_includes_overseas_filipinos',
    'gdp_total_current','gdp_total_constant','gdp_per_capita_current',
    'afs_gva_current','afs_gva_constant','derived_values','per_capita_recomputed',
    'population_total','population_source','comparable_to_2019','built_at') AS required
),
actual AS (
  SELECT collect_set(column_name) AS present
  FROM `tuklas_dev`.information_schema.columns
  WHERE table_catalog = 'tuklas_dev'
    AND table_schema = '04_gold'
    AND table_name = 'fact_province_year'
)
SELECT
  SIZE(array_except((SELECT required FROM col_list), (SELECT present FROM actual))) AS missing_count,
  (SELECT required FROM col_list) AS required_cols,
  (SELECT present FROM actual) AS actual_cols

### D5: Assert Gold validation

Evaluate all Gold validation checks. Any BLOCK failure stops the notebook before building Analytics tables.

In [0]:
%python
# D1: dim_province
d1_fails = spark.sql("SELECT fail_count FROM v_val_d1").first()["fail_count"]
assert_block("D1_dim_province", d1_fails == 0, f"{d1_fails} check(s) failed")

# D2: dim_year
d2_fails = spark.sql("SELECT fail_count FROM v_val_d2").first()["fail_count"]
assert_block("D2_dim_year", d2_fails == 0, f"{d2_fails} check(s) failed")

# D3: fact_province_year
d3_fails = spark.sql("SELECT fail_count FROM v_val_d3").first()["fail_count"]
assert_block("D3_fact_province_year", d3_fails == 0, f"{d3_fails} check(s) failed")

# D4: Required columns
d4 = spark.sql("SELECT missing_count, required_cols, actual_cols FROM v_val_d4").first()
missing = d4["missing_count"]
if missing > 0:
    required_set = set(d4["required_cols"])
    actual_set = set(d4["actual_cols"])
    missing_cols = required_set - actual_set
    assert_block("D4_required_columns", False, f"Missing columns: {missing_cols}")
else:
    assert_block("D4_required_columns", True, "All required columns present")
    # Confirm previously missing fields now exist
    for col in ['foreign_includes_overseas_filipinos', 'derived_values', 'per_capita_recomputed']:
        record_check(f"D4_{col}_exists", "REVIEW", True, f"Column {col} present in Gold fact")

print(f"\nGold validation complete: {BLOCK_COUNT} BLOCK checks, {BLOCK_FAILURES} failures, {REVIEW_COUNT} REVIEW findings")

## Phase E: Tourism activity and concentration (AQ1.1)

**Purpose:** Build `tourism_activity` (reporting unit × year) and `tourism_concentration` (year) to answer AQ1.1.

**Formulas:**
- Tourism activity = reported overnight travelers
- Tourism intensity = reported overnight travelers / population × 1,000
- Tourism share (%) = unit travelers / total eligible travelers × 100
- Top-N concentration (%) = travelers in top N eligible units / total eligible travelers × 100

**Eligibility:**
- Activity: `traveler_value_status != 'not_reported'` AND `total_travelers IS NOT NULL`
- Intensity: activity-eligible AND `population_total > 0`

**Ranks:** `RANK()` preserves ties; `province_psgc` is the deterministic tiebreaker. Ranks apply only to eligible rows.

### E1: Build tourism_activity candidate

Grain: reporting unit × year (expected 600 rows).

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_tourism_activity AS
WITH base AS (
  SELECT
    f.province_psgc, d.province_name, d.unit_type,
    d.region_psgc, d.region_name, f.year,
    f.total_travelers, f.population_total, f.population_source,
    f.traveler_value_status, f.comparable_to_2019,
    d.comparability_note, f.foreign_includes_overseas_filipinos,
    f.derived_values, f.per_capita_recomputed
  FROM v_gold_fact f
  JOIN v_gold_dim_province d ON f.province_psgc = d.province_psgc
),
elig AS (
  SELECT b.*,
    CASE WHEN traveler_value_status != 'not_reported' AND total_travelers IS NOT NULL
         THEN true ELSE false END AS activity_eligible,
    CASE WHEN traveler_value_status = 'not_reported' THEN 'Not reported by DOT'
         WHEN total_travelers IS NULL THEN 'Total travelers is NULL'
         ELSE NULL END AS activity_exclusion_reason,
    CASE WHEN traveler_value_status != 'not_reported' AND total_travelers IS NOT NULL
              AND population_total IS NOT NULL AND population_total > 0
         THEN true ELSE false END AS intensity_eligible,
    CASE WHEN traveler_value_status = 'not_reported' THEN 'Not reported by DOT'
         WHEN total_travelers IS NULL THEN 'Total travelers is NULL'
         WHEN population_total IS NULL THEN 'Population is NULL'
         WHEN population_total <= 0 THEN 'Population is nonpositive'
         ELSE NULL END AS intensity_exclusion_reason
  FROM base b
),
metrics AS (
  SELECT e.*,
    CASE WHEN intensity_eligible
         THEN CAST(total_travelers AS DOUBLE) * 1000.0 / CAST(population_total AS DOUBLE)
         ELSE NULL END AS tourism_intensity_per_1000
  FROM elig e
)
SELECT m.*,
  CASE WHEN activity_eligible
       THEN RANK() OVER (
         PARTITION BY year, CASE WHEN activity_eligible THEN 1 ELSE 0 END
         ORDER BY total_travelers DESC, province_psgc ASC)
       ELSE NULL END AS tourism_rank,
  CASE WHEN intensity_eligible
       THEN RANK() OVER (
         PARTITION BY year, CASE WHEN intensity_eligible THEN 1 ELSE 0 END
         ORDER BY tourism_intensity_per_1000 DESC, province_psgc ASC)
       ELSE NULL END AS intensity_rank,
  CASE WHEN activity_eligible
       THEN CAST(total_travelers AS DOUBLE) * 100.0
            / SUM(CASE WHEN activity_eligible THEN total_travelers ELSE 0 END) OVER (PARTITION BY year)
       ELSE NULL END AS tourism_share_pct,
  l._analytics_run_id, l._gold_fact_version, l._gold_dim_province_version, l._gold_dim_year_version, l._processed_at
FROM metrics m
CROSS JOIN v_lineage l
ORDER BY year, province_psgc

### E2: Validate tourism_activity candidate

BLOCK checks: row count, natural key, ranks/metrics only on eligible, exclusion reasons present, share sums to ~100.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_e2 AS
SELECT
  CASE WHEN COUNT(*) != 600 THEN 1 ELSE 0 END
  + CASE WHEN COUNT(*) != COUNT(DISTINCT province_psgc, year) THEN 1 ELSE 0 END
  + CASE WHEN COUNT(*) - COUNT(province_psgc) > 0 OR COUNT(*) - COUNT(year) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT activity_eligible AND tourism_rank IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT intensity_eligible AND intensity_rank IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT intensity_eligible AND tourism_intensity_per_1000 IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT activity_eligible AND tourism_share_pct IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT activity_eligible AND activity_exclusion_reason IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT intensity_eligible AND intensity_exclusion_reason IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  AS fail_count
FROM v_tourism_activity

In [0]:
%python
e2_fails = spark.sql("SELECT fail_count FROM v_val_e2").first()["fail_count"]
assert_block("E2_tourism_activity_structure", e2_fails == 0, f"{e2_fails} check(s) failed")
share_sums = spark.sql("""
  SELECT year, ROUND(SUM(tourism_share_pct), 6) AS share_sum
  FROM v_tourism_activity WHERE activity_eligible
  GROUP BY year ORDER BY year
""").collect()
for row in share_sums:
    assert_block(f"E2_share_sum_{row['year']}", abs(row['share_sum'] - 100.0) < 0.01, f"Sum={row['share_sum']:.6f}")
print("tourism_activity validation passed.")

### E4: Persist tourism_activity

In [0]:
%sql
CREATE OR REPLACE TABLE `tuklas_dev`.`05_analytics`.`tourism_activity`
USING DELTA
AS SELECT * FROM v_tourism_activity
ORDER BY year, province_psgc

In [0]:
%python
ta_count = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_activity`").first()[0]
ta_version = spark.sql("DESCRIBE HISTORY `tuklas_dev`.`05_analytics`.`tourism_activity`").orderBy("version", ascending=False).limit(1).select("version").first()[0]
record_output("tourism_activity", "SUCCESS", ta_count, int(ta_version))

### E5: Build tourism_concentration candidate

Grain: year (expected 6 rows). Top-N uses `ROW_NUMBER()` with deterministic ordering. Denominator = sum of eligible reporting-unit travelers.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_tourism_concentration AS
WITH eligible AS (
  SELECT year, province_psgc, total_travelers
  FROM v_tourism_activity WHERE activity_eligible = true
),
ranked AS (
  SELECT year, province_psgc, total_travelers,
    ROW_NUMBER() OVER (PARTITION BY year ORDER BY total_travelers DESC, province_psgc ASC) AS rn
  FROM eligible
),
year_totals AS (
  SELECT year, COUNT(*) AS eligible_reporting_unit_count,
    SUM(total_travelers) AS eligible_total_travelers
  FROM eligible GROUP BY year
),
all_units AS (
  SELECT year, COUNT(*) AS total_units FROM v_tourism_activity GROUP BY year
),
top5 AS (
  SELECT year, SUM(total_travelers) AS top_5_travelers, COUNT(*) AS top_5_unit_count
  FROM ranked WHERE rn <= 5 GROUP BY year
),
top10 AS (
  SELECT year, SUM(total_travelers) AS top_10_travelers, COUNT(*) AS top_10_unit_count
  FROM ranked WHERE rn <= 10 GROUP BY year
)
SELECT
  yt.year, yt.eligible_reporting_unit_count,
  (au.total_units - yt.eligible_reporting_unit_count) AS excluded_reporting_unit_count,
  yt.eligible_total_travelers,
  COALESCE(t5.top_5_travelers, 0) AS top_5_travelers,
  COALESCE(t5.top_5_unit_count, 0) AS top_5_unit_count,
  CASE WHEN yt.eligible_total_travelers > 0
       THEN COALESCE(t5.top_5_travelers, 0) * 100.0 / yt.eligible_total_travelers
       ELSE NULL END AS top_5_share_pct,
  COALESCE(t10.top_10_travelers, 0) AS top_10_travelers,
  COALESCE(t10.top_10_unit_count, 0) AS top_10_unit_count,
  CASE WHEN yt.eligible_total_travelers > 0
       THEN COALESCE(t10.top_10_travelers, 0) * 100.0 / yt.eligible_total_travelers
       ELSE NULL END AS top_10_share_pct,
  l._analytics_run_id, l._gold_fact_version, l._gold_dim_province_version, l._gold_dim_year_version, l._processed_at
FROM year_totals yt
LEFT JOIN all_units au ON yt.year = au.year
LEFT JOIN top5 t5 ON yt.year = t5.year
LEFT JOIN top10 t10 ON yt.year = t10.year
CROSS JOIN v_lineage l
ORDER BY yt.year

### E6: Validate and persist tourism_concentration

BLOCK checks: 6 rows, year unique, eligible + excluded = 100, shares in [0,100], top_10 >= top_5.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_e6 AS
SELECT
  CASE WHEN COUNT(*) != 6 THEN 1 ELSE 0 END
  + CASE WHEN COUNT(DISTINCT year) != COUNT(year) THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN eligible_reporting_unit_count + excluded_reporting_unit_count != 100 THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN top_5_share_pct < 0 OR top_5_share_pct > 100 THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN top_10_share_pct < 0 OR top_10_share_pct > 100 THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN top_10_share_pct < top_5_share_pct THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  AS fail_count
FROM v_tourism_concentration

In [0]:
%python
e6_fails = spark.sql("SELECT fail_count FROM v_val_e6").first()["fail_count"]
assert_block("E6_tourism_concentration", e6_fails == 0, f"{e6_fails} check(s) failed")
print("tourism_concentration validation passed.")

In [0]:
%sql
CREATE OR REPLACE TABLE `tuklas_dev`.`05_analytics`.`tourism_concentration`
USING DELTA
AS SELECT * FROM v_tourism_concentration
ORDER BY year

In [0]:
%python
tc_count = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_concentration`").first()[0]
tc_version = spark.sql("DESCRIBE HISTORY `tuklas_dev`.`05_analytics`.`tourism_concentration`").orderBy("version", ascending=False).limit(1).select("version").first()[0]
record_output("tourism_concentration", "SUCCESS", tc_count, int(tc_version))

## Phase F: Tourism recovery (AQ1.2)

**Purpose:** Build `tourism_recovery` comparing 2022 and 2023 each against 2019.

**Formula:** Recovery Index = comparison-year travelers / 2019 travelers × 100

**Classification:** `>100` → above_2019, `=100` → unchanged, `<100` → below_2019

**Eligibility:** valid 2019 baseline (total_travelers > 0, traveler_value_status != 'not_reported', comparable_to_2019 = true), comparison value present and nonnegative, comparable_to_2019 = true for comparison year. Uses exact, unrounded values for classification.

### F1: Build tourism_recovery candidate

Grain: reporting unit × comparison year (expected 200 rows = 100 × 2).

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_tourism_recovery AS
WITH baseline_2019 AS (
  SELECT province_psgc, total_travelers AS travelers_2019,
         traveler_value_status AS status_2019,
         comparable_to_2019 AS comp_2019,
         derived_values AS derived_values_2019,
         per_capita_recomputed AS per_capita_recomputed_2019
  FROM v_gold_fact WHERE year = 2019
),
comparison AS (
  SELECT province_psgc, year AS comparison_year,
         total_travelers AS travelers_comparison_year,
         traveler_value_status AS status_comparison,
         comparable_to_2019 AS comp_comparison,
         foreign_includes_overseas_filipinos,
         derived_values AS derived_values_comparison,
         per_capita_recomputed AS per_capita_recomputed_comparison
  FROM v_gold_fact WHERE year IN (2022, 2023)
),
joined AS (
  SELECT b.province_psgc, d.province_name, d.unit_type,
    d.region_psgc, d.region_name, d.comparability_note,
    c.comparison_year, b.travelers_2019, c.travelers_comparison_year,
    b.status_2019, c.status_comparison,
    b.comp_2019, c.comp_comparison,
    c.foreign_includes_overseas_filipinos,
    b.derived_values_2019, c.derived_values_comparison,
    b.per_capita_recomputed_2019, c.per_capita_recomputed_comparison
  FROM baseline_2019 b
  JOIN comparison c ON b.province_psgc = c.province_psgc
  JOIN v_gold_dim_province d ON b.province_psgc = d.province_psgc
),
elig AS (
  SELECT j.*,
    CASE WHEN travelers_2019 IS NULL THEN false
         WHEN travelers_2019 <= 0 THEN false
         WHEN status_2019 = 'not_reported' THEN false
         WHEN comp_2019 IS NULL OR comp_2019 = false THEN false
         WHEN travelers_comparison_year IS NULL THEN false
         WHEN travelers_comparison_year < 0 THEN false
         WHEN status_comparison = 'not_reported' THEN false
         WHEN comp_comparison IS NULL OR comp_comparison = false THEN false
         ELSE true END AS recovery_eligible,
    CASE WHEN travelers_2019 IS NULL THEN '2019 travelers is NULL'
         WHEN travelers_2019 <= 0 THEN '2019 travelers is zero or negative'
         WHEN status_2019 = 'not_reported' THEN '2019 not reported by DOT'
         WHEN comp_2019 IS NULL OR comp_2019 = false THEN '2019 not comparable to baseline'
         WHEN travelers_comparison_year IS NULL THEN 'Comparison year travelers is NULL'
         WHEN travelers_comparison_year < 0 THEN 'Comparison year travelers is negative'
         WHEN status_comparison = 'not_reported' THEN 'Comparison year not reported by DOT'
         WHEN comp_comparison IS NULL OR comp_comparison = false THEN 'Comparison year not comparable to 2019'
         ELSE NULL END AS recovery_exclusion_reason
  FROM joined j
)
SELECT e.*, 2019 AS baseline_year,
  (travelers_comparison_year - travelers_2019) AS traveler_change_absolute,
  CASE WHEN recovery_eligible
       THEN CAST(travelers_comparison_year AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE)
       ELSE NULL END AS recovery_index,
  CASE WHEN recovery_eligible
       THEN CASE
         WHEN CAST(travelers_comparison_year AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE) > 100 THEN 'above_2019'
         WHEN CAST(travelers_comparison_year AS DOUBLE) * 100.0 / CAST(travelers_2019 AS DOUBLE) = 100 THEN 'unchanged'
         ELSE 'below_2019' END
       ELSE NULL END AS recovery_class,
  l._analytics_run_id, l._gold_fact_version, l._gold_dim_province_version, l._gold_dim_year_version, l._processed_at
FROM elig e
CROSS JOIN v_lineage l
ORDER BY province_psgc, comparison_year

### F2: Validate and persist tourism_recovery

BLOCK checks: 200 rows, natural key unique, metrics only on eligible, ineligible have exclusion reasons, classification correctness.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_f2 AS
SELECT
  CASE WHEN COUNT(*) != 200 THEN 1 ELSE 0 END
  + CASE WHEN COUNT(*) != COUNT(DISTINCT province_psgc, comparison_year) THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN province_psgc IS NULL OR comparison_year IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT recovery_eligible AND recovery_index IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT recovery_eligible AND recovery_class IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT recovery_eligible AND recovery_exclusion_reason IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN recovery_eligible AND recovery_index IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN recovery_eligible AND recovery_class IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN recovery_eligible AND recovery_index > 100 AND recovery_class != 'above_2019' THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN recovery_eligible AND recovery_index < 100 AND recovery_class != 'below_2019' THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN recovery_eligible AND recovery_index = 100 AND recovery_class != 'unchanged' THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  AS fail_count
FROM v_tourism_recovery

In [0]:
%python
f2_fails = spark.sql("SELECT fail_count FROM v_val_f2").first()["fail_count"]
assert_block("F2_tourism_recovery", f2_fails == 0, f"{f2_fails} check(s) failed")
print("tourism_recovery validation passed.")

In [0]:
%sql
CREATE OR REPLACE TABLE `tuklas_dev`.`05_analytics`.`tourism_recovery`
USING DELTA
AS SELECT * FROM v_tourism_recovery
ORDER BY province_psgc, comparison_year

In [0]:
%python
tr_count = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_recovery`").first()[0]
tr_version = spark.sql("DESCRIBE HISTORY `tuklas_dev`.`05_analytics`.`tourism_recovery`").orderBy("version", ascending=False).limit(1).select("version").first()[0]
record_output("tourism_recovery", "SUCCESS", tr_count, int(tr_version))

## Phase G: Traveler-segment recovery (AQ1.3)

**Purpose:** Among provinces below their 2019 level in 2023, analyze recovery by traveler type (domestic, foreign, overseas Filipino).

**Headline cohort:** overall 2023 recovery eligible AND recovery_index < 100.

**Segment eligibility:**
- Domestic: computed independently (baseline > 0, comparison present and nonnegative).
- Foreign: only when `foreign_includes_overseas_filipinos` is false/NULL.
- Overseas Filipino: only when `foreign_includes_overseas_filipinos` is false/NULL.

Grain: one row per reporting unit in the 2023 below-2019 cohort.

**Coverage note:** Overseas Filipino and foreign recovery are only computed for units with a positive 2019 baseline in that segment and where `foreign_includes_overseas_filipinos` is not true. Consumers must report coverage (eligible units out of cohort size) alongside these metrics. A zero 2019 baseline does not mean no tourism in that segment.

### G1: Build traveler_segment_recovery candidate

Uses `foreign_includes_overseas_filipinos` from Gold. For units where this flag is true, foreign and overseas Filipino metrics are NULL with explicit exclusion reasons.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_traveler_segment_recovery AS
-- foreign_includes_overseas_filipinos never differs between 2019 and 2023 for any province in Gold
WITH cohort AS (
  SELECT province_psgc, recovery_index AS overall_recovery_index,
         recovery_class AS overall_recovery_class
  FROM v_tourism_recovery
  WHERE comparison_year = 2023 AND recovery_eligible = true AND recovery_index < 100
),
base_2019 AS (
  SELECT province_psgc, total_travelers AS travelers_2019,
    domestic_travelers AS dom_2019,
    foreign_travelers AS for_2019, overseas_filipinos AS of_2019,
    foreign_includes_overseas_filipinos,
    derived_values AS derived_values_2019,
    per_capita_recomputed AS per_capita_recomputed_2019
  FROM v_gold_fact WHERE year = 2019
),
comp_2023 AS (
  SELECT province_psgc, total_travelers AS travelers_2023,
    comparable_to_2019,
    domestic_travelers AS dom_2023,
    foreign_travelers AS for_2023, overseas_filipinos AS of_2023,
    derived_values AS derived_values_2023,
    per_capita_recomputed AS per_capita_recomputed_2023
  FROM v_gold_fact WHERE year = 2023
)
SELECT c.province_psgc, d.province_name, d.unit_type,
  d.region_psgc, d.region_name, d.comparability_note,
  2019 AS baseline_year, 2023 AS comparison_year,
  true AS below_2019, c.overall_recovery_index, c.overall_recovery_class,
  b.foreign_includes_overseas_filipinos,
  b.travelers_2019, co.travelers_2023, co.comparable_to_2019, -- 2023 comparison-year flag from Gold
  b.derived_values_2019, b.per_capita_recomputed_2019,
  co.derived_values_2023, co.per_capita_recomputed_2023,
  b.dom_2019 AS domestic_travelers_2019, co.dom_2023 AS domestic_travelers_2023,
  CASE WHEN b.dom_2019 IS NOT NULL AND b.dom_2019 > 0 AND co.dom_2023 IS NOT NULL AND co.dom_2023 >= 0
       THEN CAST(co.dom_2023 AS DOUBLE) * 100.0 / CAST(b.dom_2019 AS DOUBLE) ELSE NULL END AS domestic_recovery_index,
  CASE WHEN b.dom_2019 IS NOT NULL AND b.dom_2019 > 0 AND co.dom_2023 IS NOT NULL AND co.dom_2023 >= 0
       THEN CASE WHEN CAST(co.dom_2023 AS DOUBLE) * 100.0 / CAST(b.dom_2019 AS DOUBLE) > 100 THEN 'above_2019'
                 WHEN CAST(co.dom_2023 AS DOUBLE) * 100.0 / CAST(b.dom_2019 AS DOUBLE) = 100 THEN 'unchanged'
                 ELSE 'below_2019' END
       ELSE NULL END AS domestic_recovery_class,
  CASE WHEN b.dom_2019 IS NOT NULL AND b.dom_2019 > 0 AND co.dom_2023 IS NOT NULL AND co.dom_2023 >= 0
       THEN true ELSE false END AS domestic_eligible,
  CASE WHEN b.dom_2019 IS NULL THEN '2019 domestic travelers is NULL'
       WHEN b.dom_2019 <= 0 THEN '2019 domestic travelers is zero or negative'
       WHEN co.dom_2023 IS NULL THEN '2023 domestic travelers is NULL'
       WHEN co.dom_2023 < 0 THEN '2023 domestic travelers is negative'
       ELSE NULL END AS domestic_exclusion_reason,
  b.for_2019 AS foreign_travelers_2019, co.for_2023 AS foreign_travelers_2023,
  CASE WHEN b.foreign_includes_overseas_filipinos = true THEN false
       WHEN b.for_2019 IS NOT NULL AND b.for_2019 > 0 AND co.for_2023 IS NOT NULL AND co.for_2023 >= 0
       THEN true ELSE false END AS foreign_eligible,
  CASE WHEN b.foreign_includes_overseas_filipinos = true THEN 'Foreign includes overseas Filipinos for this unit'
       WHEN b.for_2019 IS NULL THEN '2019 foreign travelers is NULL'
       WHEN b.for_2019 <= 0 THEN '2019 foreign travelers is zero or negative'
       WHEN co.for_2023 IS NULL THEN '2023 foreign travelers is NULL'
       WHEN co.for_2023 < 0 THEN '2023 foreign travelers is negative'
       ELSE NULL END AS foreign_exclusion_reason,
  CASE WHEN b.foreign_includes_overseas_filipinos = true THEN NULL
       WHEN b.for_2019 IS NOT NULL AND b.for_2019 > 0 AND co.for_2023 IS NOT NULL AND co.for_2023 >= 0
       THEN CAST(co.for_2023 AS DOUBLE) * 100.0 / CAST(b.for_2019 AS DOUBLE) ELSE NULL END AS foreign_recovery_index,
  CASE WHEN b.foreign_includes_overseas_filipinos = true THEN NULL
       WHEN b.for_2019 IS NOT NULL AND b.for_2019 > 0 AND co.for_2023 IS NOT NULL AND co.for_2023 >= 0
       THEN CASE WHEN CAST(co.for_2023 AS DOUBLE) * 100.0 / CAST(b.for_2019 AS DOUBLE) > 100 THEN 'above_2019'
                 WHEN CAST(co.for_2023 AS DOUBLE) * 100.0 / CAST(b.for_2019 AS DOUBLE) = 100 THEN 'unchanged'
                 ELSE 'below_2019' END
       ELSE NULL END AS foreign_recovery_class,
  b.of_2019 AS overseas_filipino_travelers_2019, co.of_2023 AS overseas_filipino_travelers_2023,
  CASE WHEN b.foreign_includes_overseas_filipinos = true THEN false
       WHEN b.of_2019 IS NOT NULL AND b.of_2019 > 0 AND co.of_2023 IS NOT NULL AND co.of_2023 >= 0
       THEN true ELSE false END AS overseas_filipino_eligible,
  CASE WHEN b.foreign_includes_overseas_filipinos = true THEN 'Overseas Filipino included in foreign for this unit'
       WHEN b.of_2019 IS NULL THEN '2019 overseas Filipino travelers is NULL'
       WHEN b.of_2019 <= 0 THEN '2019 overseas Filipino travelers is zero or negative'
       WHEN co.of_2023 IS NULL THEN '2023 overseas Filipino travelers is NULL'
       WHEN co.of_2023 < 0 THEN '2023 overseas Filipino travelers is negative'
       ELSE NULL END AS overseas_filipino_exclusion_reason,
  CASE WHEN b.foreign_includes_overseas_filipinos = true THEN NULL
       WHEN b.of_2019 IS NOT NULL AND b.of_2019 > 0 AND co.of_2023 IS NOT NULL AND co.of_2023 >= 0
       THEN CAST(co.of_2023 AS DOUBLE) * 100.0 / CAST(b.of_2019 AS DOUBLE) ELSE NULL END AS overseas_filipino_recovery_index,
  CASE WHEN b.foreign_includes_overseas_filipinos = true THEN NULL
       WHEN b.of_2019 IS NOT NULL AND b.of_2019 > 0 AND co.of_2023 IS NOT NULL AND co.of_2023 >= 0
       THEN CASE WHEN CAST(co.of_2023 AS DOUBLE) * 100.0 / CAST(b.of_2019 AS DOUBLE) > 100 THEN 'above_2019'
                 WHEN CAST(co.of_2023 AS DOUBLE) * 100.0 / CAST(b.of_2019 AS DOUBLE) = 100 THEN 'unchanged'
                 ELSE 'below_2019' END
       ELSE NULL END AS overseas_filipino_recovery_class,
  l._analytics_run_id, l._gold_fact_version, l._gold_dim_province_version, l._gold_dim_year_version, l._processed_at
FROM cohort c
JOIN base_2019 b ON c.province_psgc = b.province_psgc
JOIN comp_2023 co ON c.province_psgc = co.province_psgc
JOIN v_gold_dim_province d ON c.province_psgc = d.province_psgc
CROSS JOIN v_lineage l
ORDER BY c.province_psgc

### G2: Validate and persist traveler_segment_recovery

BLOCK checks: natural key unique, all rows below_2019, foreign/overseas Filipino metrics NULL when foreign_includes_overseas_filipinos=true, ineligible segments have exclusion reasons.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_g2 AS
SELECT
  CASE WHEN COUNT(*) != COUNT(DISTINCT province_psgc) THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN province_psgc IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN below_2019 != true THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN overall_recovery_index IS NULL OR overall_recovery_class IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN overall_recovery_index >= 100 THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN foreign_includes_overseas_filipinos = true AND foreign_recovery_index IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN foreign_includes_overseas_filipinos = true AND overseas_filipino_recovery_index IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT domestic_eligible AND domestic_exclusion_reason IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT foreign_eligible AND foreign_exclusion_reason IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT overseas_filipino_eligible AND overseas_filipino_exclusion_reason IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN domestic_eligible AND domestic_recovery_index IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  AS fail_count
FROM v_traveler_segment_recovery

In [0]:
%python
g2_fails = spark.sql("SELECT fail_count FROM v_val_g2").first()["fail_count"]
assert_block("G2_traveler_segment_recovery", g2_fails == 0, f"{g2_fails} check(s) failed")
seg_count = spark.sql("SELECT COUNT(*) FROM v_traveler_segment_recovery").first()[0]
for_count = spark.sql("SELECT COUNT(*) FROM v_traveler_segment_recovery WHERE foreign_includes_overseas_filipinos = true").first()[0]
dom_count = spark.sql("SELECT COUNT(*) FROM v_traveler_segment_recovery WHERE domestic_eligible = true").first()[0]
print(f"Segment recovery validation passed. {seg_count} units in below-2019 cohort, {for_count} with combined foreign+overseas, {dom_count} with domestic recovery eligible.")

In [0]:
%sql
CREATE OR REPLACE TABLE `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`
USING DELTA
AS SELECT * FROM v_traveler_segment_recovery
ORDER BY province_psgc

In [0]:
%python
tsr_count = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`").first()[0]
tsr_version = spark.sql("DESCRIBE HISTORY `tuklas_dev`.`05_analytics`.`traveler_segment_recovery`").orderBy("version", ascending=False).limit(1).select("version").first()[0]
record_output("traveler_segment_recovery", "SUCCESS", tsr_count, int(tsr_version))

## Phase H: Tourism and economic alignment (BQ2)

**Purpose:** Build `tourism_economy` comparing 2019→2023 tourism change with real A&F GVA growth.

**Formulas:**
- Tourism change (%) = (2023 - 2019) / 2019 × 100
- Real A&F GVA growth (%) = (2023 constant - 2019 constant) / 2019 constant × 100 (constant-price)
- A&F share of GDP (%) = current-price A&F GVA / current-price GDP × 100
- Tourism intensity 2023 = travelers / population × 1,000

**Independent eligibility:** One invalid metric must not suppress a different valid metric. Alignment requires both tourism change and A&F growth eligible. Unchanged cases are kept separate from the four main quadrants.

**Association, not causation:** Findings describe associations — not causal relationships.

### H1: Build tourism_economy candidate

Grain: one reporting-unit row for 2019→2023 comparison (expected 100 rows). Recovery index/class reuse the computation from `v_tourism_recovery`.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_tourism_economy AS
WITH f19 AS (
  SELECT province_psgc, total_travelers AS travelers_2019,
    traveler_value_status AS status_2019, comparable_to_2019 AS comp_2019,
    afs_gva_constant AS afs_gva_constant_2019,
    gdp_per_capita_current AS gdp_per_capita_current_2019,
    derived_values AS derived_values_2019,
    per_capita_recomputed AS per_capita_recomputed_2019
  FROM v_gold_fact WHERE year = 2019
),
f23 AS (
  SELECT province_psgc, total_travelers AS travelers_2023,
    traveler_value_status AS status_2023, comparable_to_2019 AS comp_2023,
    afs_gva_constant AS afs_gva_constant_2023, afs_gva_current AS afs_gva_current_2023,
    gdp_total_current AS gdp_total_current_2023,
    gdp_per_capita_current AS gdp_per_capita_current_2023,
    population_total AS population_total_2023, population_source AS population_source_2023,
    derived_values AS derived_values_2023,
    per_capita_recomputed AS per_capita_recomputed_2023
  FROM v_gold_fact WHERE year = 2023
),
rec AS (
  SELECT province_psgc, recovery_index AS recovery_index_2023,
    recovery_class AS recovery_class_2023, recovery_eligible AS recovery_eligible_2023,
    recovery_exclusion_reason AS recovery_exclusion_reason_2023
  FROM v_tourism_recovery WHERE comparison_year = 2023
),
j AS (
  SELECT f19.province_psgc, d.province_name, d.unit_type, d.region_psgc, d.region_name,
    f19.travelers_2019, f23.travelers_2023,
    f19.status_2019, f23.status_2023, f19.comp_2019, f23.comp_2023,
    r.recovery_index_2023, r.recovery_class_2023, r.recovery_eligible_2023, r.recovery_exclusion_reason_2023,
    f19.afs_gva_constant_2019, f23.afs_gva_constant_2023,
    f23.afs_gva_current_2023, f23.gdp_total_current_2023,
    f19.gdp_per_capita_current_2019, f23.gdp_per_capita_current_2023,
    f23.population_total_2023, f23.population_source_2023,
    f19.derived_values_2019, f23.derived_values_2023,
    f19.per_capita_recomputed_2019, f23.per_capita_recomputed_2023
  FROM f19 JOIN f23 ON f19.province_psgc = f23.province_psgc
  JOIN v_gold_dim_province d ON f19.province_psgc = d.province_psgc
  LEFT JOIN rec r ON f19.province_psgc = r.province_psgc
),
elig AS (
  SELECT j.*,
    CASE WHEN COALESCE(j.recovery_eligible_2023, false) THEN true ELSE false END AS tourism_change_eligible,
    CASE WHEN j.afs_gva_constant_2019 IS NOT NULL AND j.afs_gva_constant_2023 IS NOT NULL AND j.afs_gva_constant_2019 > 0
         THEN true ELSE false END AS afs_growth_eligible,
    CASE WHEN j.afs_gva_current_2023 IS NOT NULL AND j.gdp_total_current_2023 IS NOT NULL AND j.gdp_total_current_2023 > 0
         THEN true ELSE false END AS afs_share_eligible,
    CASE WHEN j.travelers_2023 IS NOT NULL AND j.population_total_2023 IS NOT NULL AND j.population_total_2023 > 0
         THEN true ELSE false END AS tourism_intensity_eligible,
    CASE WHEN COALESCE(j.recovery_eligible_2023, false) AND j.afs_gva_constant_2019 IS NOT NULL
              AND j.afs_gva_constant_2023 IS NOT NULL AND j.afs_gva_constant_2019 > 0
         THEN true ELSE false END AS alignment_eligible
  FROM j
)
SELECT e.*, 2019 AS baseline_year, 2023 AS comparison_year,
  CASE WHEN tourism_change_eligible THEN (CAST(travelers_2023 AS DOUBLE) - CAST(travelers_2019 AS DOUBLE)) * 100.0 / CAST(travelers_2019 AS DOUBLE) ELSE NULL END AS tourism_change_pct,
  CASE WHEN afs_growth_eligible THEN (CAST(afs_gva_constant_2023 AS DOUBLE) - CAST(afs_gva_constant_2019 AS DOUBLE)) * 100.0 / CAST(afs_gva_constant_2019 AS DOUBLE) ELSE NULL END AS afs_gva_growth_pct,
  CASE WHEN tourism_change_eligible THEN CASE WHEN travelers_2023 > travelers_2019 THEN 'up' WHEN travelers_2023 = travelers_2019 THEN 'unchanged' ELSE 'down' END ELSE NULL END AS tourism_direction,
  CASE WHEN afs_growth_eligible THEN CASE WHEN afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'up' WHEN afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'unchanged' ELSE 'down' END ELSE NULL END AS economy_direction,
  CASE WHEN alignment_eligible THEN
    CASE WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'tourism_up_economy_up'
         WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'tourism_up_economy_down'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'tourism_down_economy_up'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'tourism_down_economy_down'
         WHEN travelers_2023 = travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'tourism_unchanged_economy_up'
         WHEN travelers_2023 = travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'tourism_unchanged_economy_down'
         WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'tourism_up_economy_unchanged'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 = afs_gva_constant_2019 THEN 'tourism_down_economy_unchanged'
         ELSE 'tourism_unchanged_economy_unchanged' END
    ELSE NULL END AS tourism_economy_pattern,
  CASE WHEN alignment_eligible THEN
    CASE WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'Tourism activity increased; real A&F GVA increased'
         WHEN travelers_2023 > travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'Tourism activity increased; real A&F GVA decreased'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 > afs_gva_constant_2019 THEN 'Tourism activity decreased; real A&F GVA increased'
         WHEN travelers_2023 < travelers_2019 AND afs_gva_constant_2023 < afs_gva_constant_2019 THEN 'Tourism activity decreased; real A&F GVA decreased'
         ELSE 'Tourism or A&F GVA unchanged' END
    ELSE NULL END AS tourism_economy_pattern_label,
  CASE WHEN afs_share_eligible THEN CAST(afs_gva_current_2023 AS DOUBLE) * 100.0 / CAST(gdp_total_current_2023 AS DOUBLE) ELSE NULL END AS afs_share_of_gdp_pct,
  CASE WHEN tourism_intensity_eligible THEN CAST(travelers_2023 AS DOUBLE) * 1000.0 / CAST(population_total_2023 AS DOUBLE) ELSE NULL END AS tourism_intensity_per_1000_2023,
  CASE WHEN NOT tourism_change_eligible THEN COALESCE(recovery_exclusion_reason_2023, 'Tourism change not eligible') ELSE NULL END AS tourism_change_exclusion_reason,
  CASE WHEN NOT afs_growth_eligible THEN CASE WHEN afs_gva_constant_2019 IS NULL THEN '2019 A&F GVA constant is NULL' WHEN afs_gva_constant_2023 IS NULL THEN '2023 A&F GVA constant is NULL' WHEN afs_gva_constant_2019 <= 0 THEN '2019 A&F GVA constant is zero or negative' ELSE 'Unknown' END ELSE NULL END AS afs_growth_exclusion_reason,
  CASE WHEN NOT afs_share_eligible THEN CASE WHEN afs_gva_current_2023 IS NULL THEN '2023 A&F GVA current is NULL' WHEN gdp_total_current_2023 IS NULL THEN '2023 GDP current is NULL' WHEN gdp_total_current_2023 <= 0 THEN '2023 GDP current is zero or negative' ELSE 'Unknown' END ELSE NULL END AS afs_share_exclusion_reason,
  CASE WHEN NOT tourism_intensity_eligible THEN CASE WHEN travelers_2023 IS NULL THEN '2023 travelers is NULL' WHEN population_total_2023 IS NULL THEN '2023 population is NULL' WHEN population_total_2023 <= 0 THEN '2023 population is nonpositive' ELSE 'Unknown' END ELSE NULL END AS tourism_intensity_exclusion_reason,
  CASE WHEN NOT alignment_eligible THEN CASE WHEN NOT tourism_change_eligible AND NOT afs_growth_eligible THEN 'Both tourism change and A&F growth not eligible' WHEN NOT tourism_change_eligible THEN 'Tourism change not eligible' ELSE 'A&F growth not eligible' END ELSE NULL END AS alignment_exclusion_reason,
  l._analytics_run_id, l._gold_fact_version, l._gold_dim_province_version, l._gold_dim_year_version, l._processed_at
FROM elig e
CROSS JOIN v_lineage l
ORDER BY province_psgc

### H2: Validate and persist tourism_economy

BLOCK checks: 100 rows, natural key unique, metrics only on eligible rows, ineligible have exclusion reasons, pattern correctness.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_h2 AS
SELECT
  CASE WHEN COUNT(*) != 100 THEN 1 ELSE 0 END
  + CASE WHEN COUNT(*) != COUNT(DISTINCT province_psgc) THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN province_psgc IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT tourism_change_eligible AND tourism_change_pct IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT afs_growth_eligible AND afs_gva_growth_pct IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT afs_share_eligible AND afs_share_of_gdp_pct IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT tourism_intensity_eligible AND tourism_intensity_per_1000_2023 IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT alignment_eligible AND tourism_economy_pattern IS NOT NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN alignment_eligible AND tourism_economy_pattern IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT tourism_change_eligible AND tourism_change_exclusion_reason IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT afs_growth_eligible AND afs_growth_exclusion_reason IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  + CASE WHEN SUM(CASE WHEN NOT afs_share_eligible AND afs_share_exclusion_reason IS NULL THEN 1 ELSE 0 END) > 0 THEN 1 ELSE 0 END
  AS fail_count
FROM v_tourism_economy

In [0]:
%python
h2_fails = spark.sql("SELECT fail_count FROM v_val_h2").first()["fail_count"]
assert_block("H2_tourism_economy", h2_fails == 0, f"{h2_fails} check(s) failed")
# REVIEW: check for A&F share > 100%
over_100 = spark.sql("SELECT COUNT(*) FROM v_tourism_economy WHERE afs_share_of_gdp_pct > 100").first()[0]
if over_100 > 0:
    record_check("H2_afs_share_over_100", "REVIEW", False, f"{over_100} units with A&F share > 100% — investigate")
else:
    record_check("H2_afs_share_over_100", "REVIEW", True, "No units with A&F share > 100%")
print("tourism_economy validation passed.")

In [0]:
%sql
CREATE OR REPLACE TABLE `tuklas_dev`.`05_analytics`.`tourism_economy`
USING DELTA
AS SELECT * FROM v_tourism_economy
ORDER BY province_psgc

In [0]:
%python
te_count = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`tourism_economy`").first()[0]
te_version = spark.sql("DESCRIBE HISTORY `tuklas_dev`.`05_analytics`.`tourism_economy`").orderBy("version", ascending=False).limit(1).select("version").first()[0]
record_output("tourism_economy", "SUCCESS", te_count, int(te_version))

## Phase I: Cross-table consistency

**Purpose:** Verify consistency between Analytics outputs using null-safe comparisons.

Checks:
1. `tourism_economy.recovery_index_2023` = `tourism_recovery.recovery_index` where `comparison_year = 2023`
2. `tourism_economy.recovery_class_2023` = `tourism_recovery.recovery_class` where `comparison_year = 2023`
3. `tourism_economy.tourism_intensity_per_1000_2023` = `tourism_activity.tourism_intensity_per_1000` where `year = 2023`
4. `tourism_concentration` denominator = sum of eligible `tourism_activity` travelers

In [0]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_val_i1 AS
WITH econ_vs_rec AS (
  SELECT COUNT(*) AS mismatches
  FROM `tuklas_dev`.`05_analytics`.`tourism_economy` e
  JOIN `tuklas_dev`.`05_analytics`.`tourism_recovery` r
    ON e.province_psgc = r.province_psgc AND r.comparison_year = 2023
  WHERE NOT (e.recovery_index_2023 <=> r.recovery_index)
     OR NOT (e.recovery_class_2023 <=> r.recovery_class)
),
econ_vs_act AS (
  SELECT COUNT(*) AS mismatches
  FROM `tuklas_dev`.`05_analytics`.`tourism_economy` e
  JOIN `tuklas_dev`.`05_analytics`.`tourism_activity` a
    ON e.province_psgc = a.province_psgc AND a.year = 2023
  WHERE NOT (e.tourism_intensity_per_1000_2023 <=> a.tourism_intensity_per_1000)
),
conc_vs_act AS (
  SELECT COUNT(*) AS mismatches
  FROM `tuklas_dev`.`05_analytics`.`tourism_concentration` c
  JOIN (
    SELECT year, SUM(total_travelers) AS eligible_sum
    FROM `tuklas_dev`.`05_analytics`.`tourism_activity`
    WHERE activity_eligible = true
    GROUP BY year
  ) a ON c.year = a.year
  WHERE NOT (c.eligible_total_travelers <=> a.eligible_sum)
)
SELECT
  (SELECT mismatches FROM econ_vs_rec) AS econ_rec_mismatches,
  (SELECT mismatches FROM econ_vs_act) AS econ_act_mismatches,
  (SELECT mismatches FROM conc_vs_act) AS conc_act_mismatches

In [0]:
%python
i1 = spark.sql("SELECT * FROM v_val_i1").first()
assert_block("I1_econ_vs_recovery", i1["econ_rec_mismatches"] == 0, f"{i1['econ_rec_mismatches']} mismatches")
assert_block("I1_econ_vs_activity_intensity", i1["econ_act_mismatches"] == 0, f"{i1['econ_act_mismatches']} mismatches")
assert_block("I1_concentration_vs_activity_sum", i1["conc_act_mismatches"] == 0, f"{i1['conc_act_mismatches']} mismatches")
print("Cross-table consistency validation passed.")

## Phase J: Execution summary

**Purpose:** Evidence-based execution summary reporting actual results — not hardcoded success statements. Reports Gold versions, table statuses, check counts, traveler-segment support, and overall build result.

In [0]:
%python
# ── Execution Summary ──────────────────────────────────────────────────────
print("=" * 80)
print(f"ANALYTICS BUILD EXECUTION SUMMARY")
print(f"Run ID: {run_id}")
print(f"Notebook: {NOTEBOOK_NAME}")
print("=" * 80)

# Gold versions
print(f"\nGold Delta versions used:")
for tbl, v in gold_versions.items():
    print(f"  {tbl}: v{v}")

# Table statuses
print(f"\nOutput tables:")
tables_expected = ['tourism_activity', 'tourism_concentration', 'tourism_recovery',
                  'traveler_segment_recovery', 'tourism_economy']
success_count = 0
fail_count = 0
not_attempted = 0
for tbl in tables_expected:
    if tbl in output_status:
        st = output_status[tbl]
        print(f"  {tbl}: {st['status']} (rows={st['row_count']}, delta_v={st['delta_version']})")
        if st['status'] == 'SUCCESS':
            success_count += 1
        else:
            fail_count += 1
    else:
        print(f"  {tbl}: NOT ATTEMPTED")
        not_attempted += 1

print(f"\nTables: {success_count} success, {fail_count} failed, {not_attempted} not attempted")

# Re-read Gold versions to detect concurrent writes (run before check counts so REVIEW is counted)
print(f"\nGold version stability check:")
gold_versions_end = {}
for tbl in ['dim_province', 'dim_year', 'fact_province_year']:
    try:
        v = spark.sql(f"DESCRIBE HISTORY `{CATALOG}`.`{GOLD_SCHEMA}`.`{tbl}`").orderBy("version", ascending=False).limit(1).select("version").first()[0]
        gold_versions_end[tbl] = int(v)
    except:
        gold_versions_end[tbl] = None
versions_unchanged = all(gold_versions_end.get(t) == gold_versions.get(t) for t in ['dim_province', 'dim_year', 'fact_province_year'])
record_check("Gold_versions_unchanged_during_run", "REVIEW", versions_unchanged,
    f"Start: {gold_versions} | End: {gold_versions_end}")

# Check summary
print(f"\nValidation checks:")
print(f"  BLOCK checks: {BLOCK_COUNT}")
print(f"  BLOCK failures: {BLOCK_FAILURES}")
print(f"  REVIEW findings: {REVIEW_COUNT}")

# Traveler-segment support
print(f"\nTraveler-segment support:")
if 'traveler_segment_recovery' in output_status:
    tsr_rows = output_status['traveler_segment_recovery']['row_count']
    for_combined = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery` WHERE foreign_includes_overseas_filipinos = true").first()[0]
    dom_elig = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery` WHERE domestic_eligible = true").first()[0]
    for_elig = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery` WHERE foreign_eligible = true").first()[0]
    of_elig = spark.sql("SELECT COUNT(*) FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery` WHERE overseas_filipino_eligible = true").first()[0]
    print(f"  Below-2019 cohort size: {tsr_rows}")
    print(f"  Domestic: {dom_elig}/{tsr_rows} ({dom_elig/tsr_rows*100:.1f}%)")
    print(f"  Foreign: {for_elig}/{tsr_rows} ({for_elig/tsr_rows*100:.1f}%) (blocked for {for_combined} units with foreign_includes_overseas_filipinos)")
    print(f"  Overseas Filipino: {of_elig}/{tsr_rows} ({of_elig/tsr_rows*100:.1f}%)")
    # Exclusion reason breakdown
    for seg, col in [("Domestic", "domestic_exclusion_reason"), ("Foreign", "foreign_exclusion_reason"), ("Overseas Filipino", "overseas_filipino_exclusion_reason")]:
        reasons = spark.sql(f"SELECT `{col}` AS reason, COUNT(*) AS cnt FROM `tuklas_dev`.`05_analytics`.`traveler_segment_recovery` GROUP BY `{col}` ORDER BY cnt DESC").collect()
        print(f"  {seg} exclusion reasons:")
        for r in reasons:
            label = r['reason'] if r['reason'] else '(eligible)'
            print(f"    {label}: {r['cnt']}")
else:
    print("  traveler_segment_recovery not persisted")

# Gold field status
print(f"\nGold field status:")
for col in ['foreign_includes_overseas_filipinos', 'derived_values', 'per_capita_recomputed']:
    exists = col in [r.column_name for r in spark.sql("SELECT column_name FROM `tuklas_dev`.information_schema.columns WHERE table_schema='04_gold' AND table_name='fact_province_year'").collect()]
    print(f"  {col}: {'present' if exists else 'MISSING'}")

# Traceability
print(f"\nTraceability:")
print(f"  Analytics run_id: {run_id}")
print(f"  Gold fact version: {gold_versions.get('fact_province_year')}")
print(f"  Gold dim_province version: {gold_versions.get('dim_province')}")
print(f"  Gold dim_year version: {gold_versions.get('dim_year')}")
print(f"  Processing timestamp: {processed_at}")

# Overall result
if BLOCK_FAILURES > 0:
    overall = "FAILED"
elif fail_count > 0:
    overall = "PARTIAL_FAILURE"
elif REVIEW_COUNT > 0:
    overall = "SUCCESS_WITH_REVIEW"
else:
    overall = "SUCCESS"

print(f"\n{'=' * 80}")
print(f"OVERALL BUILD RESULT: {overall}")
if fail_count > 0 and success_count > 0:
    print(f"PARTIAL COMPLETION: {success_count} of {len(tables_expected)} tables written successfully")
print(f"{'=' * 80}")